[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/065.%20The%203D%20Pallet_Case%20Packing%20Problem/P65-Tier-5_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 65. The 3D Pallet/Case Packing Problem

## Tier 5: An Integrated Digital-Twin Simulation

### Purpose and learning objectives

This notebook demonstrates how a warehouse-level simulation can connect 3D packing tasks with operational data and decisions. It models a four-layer digital-twin architecture—physical assets, connectivity, data processing, and applications—and includes simplified representations of warehouse, transportation, labor, and demand systems.

By the end, you should be able to trace how simulated sensor data reaches a decision rule, how operating conditions select a packing policy, how a geometric feasibility check prevents item overlap, and how simulated KPIs can be summarized and compared with an explicitly assumed baseline.

### Scope and modeling assumptions

- This is a self-contained, stochastic teaching simulation. It does not connect to warehouse hardware, external databases, or production management systems. Sensor readings, operational decisions, and KPI values are generated by Python code.
- Packing uses integer dimensions and integer-coordinate positions. Items are axis-aligned, keep their listed orientation, and are placed into one bin. The greedy procedure orders items by decreasing volume and searches candidate coordinates; it is a heuristic, not a proof of optimality. Items that cannot be placed remain unpacked.
- The named packing strategies are rule-selected policies, not distinct optimization algorithms. In this demonstration, several policies call the same greedy packing procedure. Strategy names therefore describe the decision rule's intent, not a measured performance guarantee.
- Scenario configurations alter the simulated warehouse configuration, but the model does not establish causal effects of those changes on performance. KPI values are independently simulated.
- The simulation does not implement machine learning, live synchronization, an economic return-on-investment model, or a validated operational digital twin. Those topics require measured data and additional model components.

### Solution sequence

1. Generate simulated physical-layer sensor readings.
2. Pass readings through connectivity, data-processing, and application layers.
3. Let the application layer generate operational decisions and packing tasks.
4. Select a packing policy from the current conditions, then apply the geometric placement checks.
5. Record simulated KPIs and use the same simulation structure to examine configured what-if scenarios.

The central geometric test is whether two item interiors overlap. For items with lower-corner coordinates \((x_i,y_i,z_i)\) and dimensions \((l_i,w_i,h_i)\), they overlap only when their projections overlap along all three axes:

$$\begin{aligned}
O_{ij} ={}& [x_i < x_j + l_j] [x_j < x_i + l_i] \\
          &\cdot [y_i < y_j + w_j] [y_j < y_i + w_i] \\
          &\cdot [z_i < z_j + h_j] [z_j < z_i + h_i].
\end{aligned}$$

Here, each bracket is 1 when its inequality is true and 0 otherwise. The items overlap exactly when \(O_{ij}=1\). The code expresses this test by checking whether the intervals are separated on any axis. It also checks that each item's upper corner remains within the bin. These tests ensure that every accepted placement is geometrically feasible with respect to boundaries and overlap; they do not ensure that the heuristic packs every item or finds a best arrangement.

### Interpreting results

Randomly generated KPI values illustrate how to collect and summarize simulation outputs; they are not evidence of real warehouse performance. The strategy demonstration shows which policy the rules select under specified conditions. The baseline comparison is illustrative because its baseline values are assumptions rather than a control group. Treat annualized cost figures as extrapolations under stated operating assumptions, not measured savings.

In [ ]:
import random
import time
from copy import deepcopy
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np

# A fixed seed makes this demonstration reproducible when run from the top.
random.seed(42)
np.random.seed(42)

In [ ]:
class DigitalTwin3DPacking:
    """Simulate integrated warehouse operations and single-bin 3D packing."""

    def __init__(self, warehouse_config, simulation_hours=24):
        self.warehouse_config = warehouse_config
        self.simulation_hours = simulation_hours
        self.current_time = datetime(2024, 1, 1, 8, 0, 0)

        self.physical_layer = PhysicalAssetsLayer(warehouse_config)
        self.connectivity_layer = ConnectivityLayer()
        self.data_processing_layer = DataProcessingLayer()
        self.application_layer = ApplicationLayer()

        self.wms = WarehouseManagementSystem()
        self.tms = TransportationManagementSystem()
        self.lms = LaborManagementSystem()
        self.dfs = DemandForecastingSystem()

        self.iot_sensors = IoTSensorNetwork(warehouse_config)
        self.rfid_trackers = RFIDTrackingSystem()
        self.automated_equipment = AutomatedEquipmentSystem()

        self.demand_predictor = DemandPredictor()
        self.disruption_predictor = DisruptionPredictor()
        self.kpi_monitor = KPIMonitor()
        self.performance_history = []
        self.scenario_analyzer = ScenarioAnalyzer()

        print(f"Digital twin initialized for {warehouse_config['name']}.")
        print(
            f"Simulation period: {simulation_hours} hours, "
            f"starting at {self.current_time}."
        )

    def run_simulation(self):
        """Run the simulated warehouse process one hour at a time."""
        print("\n" + "=" * 60)
        print("STARTING DIGITAL-TWIN SIMULATION")
        print("=" * 60)

        for hour_counter in range(self.simulation_hours):
            print(
                f"\n--- Hour {hour_counter + 1}: "
                f"{self.current_time.strftime('%H:%M')} ---"
            )

            sensor_data = self.physical_layer.collect_sensor_data(
                self.current_time
            )
            transmitted_data = self.connectivity_layer.transmit_data(
                sensor_data
            )
            processed_data = self.data_processing_layer.process_data(
                transmitted_data
            )
            system_decisions = self.application_layer.execute_logic(
                processed_data
            )

            self._execute_integrated_operations(system_decisions)
            self._update_predictive_analytics()
            self._monitor_performance()
            self.current_time += timedelta(hours=1)

        print("\n" + "=" * 60)
        print("SIMULATION COMPLETED")
        print("=" * 60)
        return self.performance_history

    def _execute_integrated_operations(self, decisions):
        """Pass simulated decisions to the connected system models."""
        wms_actions = self.wms.process_orders(
            decisions['orders'], decisions['inventory']
        )
        tms_actions = self.tms.schedule_shipments(
            decisions['shipments'], decisions['vehicles']
        )
        lms_actions = self.lms.allocate_workers(
            decisions['workload'], decisions['available_workers']
        )
        dfs_actions = self.dfs.update_forecast(
            decisions['demand_patterns'], decisions['seasonal_factors']
        )
        packing_results = self._execute_packing_operations(
            decisions['packing_tasks']
        )
        return {
            'wms': wms_actions,
            'tms': tms_actions,
            'lms': lms_actions,
            'dfs': dfs_actions,
            'packing': packing_results,
        }

    def _execute_packing_operations(self, packing_tasks):
        """Select a policy and pack each task using its single-bin model."""
        results = []
        for task in packing_tasks:
            conditions = self._get_current_operational_conditions()
            strategy = self._select_packing_strategy(task, conditions)
            packing_result = self._pack_items(
                task['items'], task['bin_dims'], strategy
            )
            results.append(
                {
                    'task_id': task['id'],
                    'strategy': strategy,
                    'result': packing_result,
                    'conditions': conditions,
                    'timestamp': self.current_time,
                }
            )
        return results

    def _get_current_operational_conditions(self):
        """Collect the modeled operating indicators used by the policy rule."""
        return {
            'labor_availability': self.lms.get_available_workers(),
            'equipment_status': self.automated_equipment.get_status(),
            'congestion_level': self.wms.get_congestion_level(),
            'demand_pressure': self.dfs.get_demand_pressure(),
            'transport_capacity': self.tms.get_available_capacity(),
            'inventory_levels': self.wms.get_inventory_levels(),
        }

    def _select_packing_strategy(self, task, conditions):
        """Apply a priority-ordered rule to select a named packing policy."""
        if conditions['demand_pressure'] > 0.8:
            return 'speed_optimized'
        if conditions['labor_availability'] < 0.5:
            return 'automation_focused'
        if conditions['congestion_level'] > 0.7:
            return 'space_efficient'
        if conditions['transport_capacity'] < 0.6:
            return 'container_optimized'
        return 'balanced'

    def _pack_items(self, items, bin_dims, strategy):
        """Dispatch to the packing routine associated with the policy name."""
        strategy_methods = {
            'speed_optimized': self._speed_optimized_packing,
            'automation_focused': self._automation_focused_packing,
            'space_efficient': self._space_efficient_packing,
            'container_optimized': self._container_optimized_packing,
            'balanced': self._balanced_packing,
        }
        if strategy not in strategy_methods:
            raise ValueError(f"Unknown packing strategy: {strategy}")
        return strategy_methods[strategy](items, bin_dims)

    def _speed_optimized_packing(self, items, bin_dims):
        solution = self._greedy_packing(items, bin_dims)
        solution['strategy'] = 'speed_optimized'
        return solution

    def _automation_focused_packing(self, items, bin_dims):
        solution = self._greedy_packing(items, bin_dims)
        solution['strategy'] = 'automation_focused'
        solution['automation_compatible'] = True
        return solution

    def _space_efficient_packing(self, items, bin_dims):
        solution = self._improved_packing(items, bin_dims)
        solution['strategy'] = 'space_efficient'
        return solution

    def _container_optimized_packing(self, items, bin_dims):
        solution = self._container_minimizing_packing(items, bin_dims)
        solution['strategy'] = 'container_optimized'
        return solution

    def _balanced_packing(self, items, bin_dims):
        solution = self._improved_packing(items, bin_dims)
        solution['strategy'] = 'balanced'
        return solution

    def _greedy_packing(self, items, bin_dims):
        """Place larger-volume items first at the first feasible grid point."""
        length, width, height = bin_dims
        solution = {
            'positions': [],
            'placed_item_indices': [],
            'bins_used': 1,
            'items_packed': 0,
            'total_volume': sum(l * w * h for l, w, h in items),
        }
        item_indices = sorted(
            range(len(items)),
            key=lambda i: items[i][0] * items[i][1] * items[i][2],
            reverse=True,
        )

        for item_idx in item_indices:
            item_length, item_width, item_height = items[item_idx]
            placed = False
            for x in range(length - item_length + 1):
                for y in range(width - item_width + 1):
                    for z in range(height - item_height + 1):
                        position = (x, y, z)
                        if self._is_position_feasible(
                            item_idx, position, solution, items, bin_dims
                        ):
                            solution['positions'].append(position)
                            solution['placed_item_indices'].append(item_idx)
                            solution['items_packed'] += 1
                            placed = True
                            break
                    if placed:
                        break
                if placed:
                    break
        return solution

    def _improved_packing(self, items, bin_dims):
        """Return the demonstration's greedy solution and volume utilization."""
        solution = self._greedy_packing(items, bin_dims)
        bin_volume = bin_dims[0] * bin_dims[1] * bin_dims[2]
        packed_volume = sum(
            items[i][0] * items[i][1] * items[i][2]
            for i in solution['placed_item_indices']
        )
        solution['utilization'] = packed_volume / bin_volume
        return solution

    def _container_minimizing_packing(self, items, bin_dims):
        """Apply the available single-bin heuristic for this named policy."""
        return self._improved_packing(items, bin_dims)

    def _is_position_feasible(
        self, item_idx, position, solution, items, bin_dims
    ):
        """Check bin boundaries and non-overlap with every placed item."""
        x, y, z = position
        length, width, height = items[item_idx]
        bin_length, bin_width, bin_height = bin_dims

        if (
            x < 0
            or y < 0
            or z < 0
            or x + length > bin_length
            or y + width > bin_width
            or z + height > bin_height
        ):
            return False

        for placed_idx, placed_position in zip(
            solution['placed_item_indices'], solution['positions']
        ):
            x2, y2, z2 = placed_position
            length2, width2, height2 = items[placed_idx]
            overlap_x = x < x2 + length2 and x2 < x + length
            overlap_y = y < y2 + width2 and y2 < y + width
            overlap_z = z < z2 + height2 and z2 < z + height
            if overlap_x and overlap_y and overlap_z:
                return False
        return True

    def _update_predictive_analytics(self):
        demand_data = self.dfs.get_historical_demand()
        self.demand_predictor.update_model(demand_data)
        operational_data = self._get_current_operational_conditions()
        self.disruption_predictor.update_model(operational_data)

    def _monitor_performance(self):
        kpis = self.kpi_monitor.calculate_kpis(self.current_time)
        self.performance_history.append(
            {'timestamp': self.current_time, 'kpis': kpis}
        )

    def run_what_if_analysis(self, scenarios):
        print("\n" + "=" * 60)
        print("WHAT-IF SCENARIO ANALYSIS")
        print("=" * 60)
        results = {}
        for scenario_name, scenario_params in scenarios.items():
            print(f"\nAnalyzing scenario: {scenario_name}")
            scenario_twin = self._create_scenario_twin(scenario_params)
            scenario_results = scenario_twin.run_simulation()
            analysis = self.scenario_analyzer.analyze_results(scenario_results)
            results[scenario_name] = {
                'parameters': scenario_params,
                'results': scenario_results,
                'analysis': analysis,
            }
            print(f"Scenario {scenario_name} completed.")
        return results

    def _create_scenario_twin(self, scenario_params):
        scenario_config = deepcopy(self.warehouse_config)
        for key, value in scenario_params.items():
            if key in scenario_config:
                scenario_config[key] = value
        return DigitalTwin3DPacking(scenario_config, simulation_hours=4)

    def visualize_performance(self):
        """Plot recorded KPI series and a normalized KPI summary."""
        if not self.performance_history:
            print("No performance data is available to visualize.")
            return

        timestamps = [entry['timestamp'] for entry in self.performance_history]
        kpi_data = [entry['kpis'] for entry in self.performance_history]
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
            2, 2, figsize=(15, 10)
        )

        order_rates = [kpi['orders_processed_per_hour'] for kpi in kpi_data]
        ax1.plot(timestamps, order_rates, color='blue', marker='o')
        ax1.set_xlabel('Time')
        ax1.set_ylabel('Orders per hour')
        ax1.set_title('Order Processing Rate')
        ax1.grid(True, alpha=0.3)
        ax1.tick_params(axis='x', rotation=45)

        utilizations = [kpi['space_utilization'] for kpi in kpi_data]
        ax2.plot(timestamps, utilizations, color='green', marker='s')
        ax2.set_xlabel('Time')
        ax2.set_ylabel('Utilization (fraction)')
        ax2.set_title('Space Utilization')
        ax2.grid(True, alpha=0.3)
        ax2.tick_params(axis='x', rotation=45)

        labor_efficiency = [kpi['labor_efficiency'] for kpi in kpi_data]
        ax3.plot(timestamps, labor_efficiency, color='red', marker='^')
        ax3.set_xlabel('Time')
        ax3.set_ylabel('Efficiency (fraction)')
        ax3.set_title('Labor Efficiency')
        ax3.grid(True, alpha=0.3)
        ax3.tick_params(axis='x', rotation=45)

        categories = ['Speed', 'Quality', 'Efficiency', 'Cost', 'Reliability']
        avg_order_rate = np.mean(order_rates)
        avg_scores = [
            min(avg_order_rate / 200, 1.0),
            np.mean([kpi['packing_quality'] for kpi in kpi_data]),
            np.mean(labor_efficiency),
            1 - (np.mean([kpi['cost_per_order'] for kpi in kpi_data]) / 10),
            np.mean([kpi['system_uptime'] for kpi in kpi_data]),
        ]
        values = avg_scores + avg_scores[:1]
        angles = np.linspace(
            0, 2 * np.pi, len(categories), endpoint=False
        ).tolist()
        angles += angles[:1]
        ax4.plot(angles, values, marker='o', color='purple')
        ax4.fill(angles, values, alpha=0.25, color='purple')
        ax4.set_xticks(angles[:-1])
        ax4.set_xticklabels(categories)
        ax4.set_ylim(0, 1)
        ax4.set_title('Normalized KPI Summary')
        ax4.grid(True)

        plt.tight_layout()
        plt.show()
        self._print_performance_summary(kpi_data)

    def _print_performance_summary(self, kpi_data):
        """Print means and a clearly defined illustrative cost comparison."""
        print("\n" + "=" * 60)
        print("SIMULATED PERFORMANCE SUMMARY")
        print("=" * 60)
        mean_order_rate = np.mean(
            [kpi['orders_processed_per_hour'] for kpi in kpi_data]
        )
        mean_utilization = np.mean(
            [kpi['space_utilization'] for kpi in kpi_data]
        )
        mean_labor_efficiency = np.mean(
            [kpi['labor_efficiency'] for kpi in kpi_data]
        )
        mean_cost = np.mean([kpi['cost_per_order'] for kpi in kpi_data])
        mean_uptime = np.mean(
            [kpi['system_uptime'] for kpi in kpi_data]
        )
        print(f"Mean order processing rate: {mean_order_rate:.1f} orders/hour")
        print(f"Mean space utilization: {mean_utilization:.1%}")
        print(f"Mean labor efficiency: {mean_labor_efficiency:.1%}")
        print(f"Mean simulated cost per order: ${mean_cost:.2f}")
        print(f"Mean simulated system uptime: {mean_uptime:.1%}")

        if len(kpi_data) > 1:
            first_rate = kpi_data[0]['orders_processed_per_hour']
            last_rate = kpi_data[-1]['orders_processed_per_hour']
            first_utilization = kpi_data[0]['space_utilization']
            last_utilization = kpi_data[-1]['space_utilization']
            rate_change = (last_rate - first_rate) / first_rate * 100
            utilization_change = (
                (last_utilization - first_utilization)
                / first_utilization
                * 100
            )
            print("\nFirst-to-last simulated KPI change:")
            print(f"  Order processing rate: {rate_change:+.1f}%")
            print(f"  Space utilization: {utilization_change:+.1f}%")

        total_orders = sum(
            kpi['orders_processed_per_hour'] for kpi in kpi_data
        )
        actual_cost = sum(
            kpi['orders_processed_per_hour'] * kpi['cost_per_order']
            for kpi in kpi_data
        )
        baseline_cost_per_order = 5.0
        baseline_cost = total_orders * baseline_cost_per_order
        savings = baseline_cost - actual_cost
        print("\nIllustrative cost comparison:")
        print(f"  Simulated orders: {total_orders:.0f}")
        print(
            f"  Assumed baseline cost at ${baseline_cost_per_order:.2f} "
            f"per order: ${baseline_cost:.2f}"
        )
        print(f"  Simulated cost total: ${actual_cost:.2f}")
        print(
            f"  Difference relative to assumed baseline: ${savings:.2f}"
        )


class PhysicalAssetsLayer:
    """Generate illustrative readings for modeled physical assets."""

    def __init__(self, config):
        self.config = config
        self.sensors = config.get('sensors', 500)
        self.conveyors = config.get('conveyors', 20)
        self.robots = config.get('robots', 10)

    def collect_sensor_data(self, timestamp):
        return {
            'temperature': random.uniform(18, 25),
            'humidity': random.uniform(40, 60),
            'equipment_status': random.choice(
                ['operational', 'maintenance', 'idle']
            ),
            'congestion_level': random.uniform(0.3, 0.9),
            'timestamp': timestamp,
        }


class ConnectivityLayer:
    """Represent data transmission with simple simulated characteristics."""

    def __init__(self):
        self.bandwidth = 1000
        self.latency = 5

    def transmit_data(self, data):
        return {
            'data': data,
            'transmission_time': self.latency,
            'packet_loss': random.uniform(0.001, 0.01),
        }


class DataProcessingLayer:
    """Pass transmitted readings through the simulated data layer."""

    def __init__(self):
        self.processing_power = 1000

    def process_data(self, transmitted_data):
        return {
            'processed_data': transmitted_data['data'],
            'processing_time': 0.1,
            'data_quality': random.uniform(0.95, 0.99),
        }


class ApplicationLayer:
    """Generate simulated decisions and packing tasks for each time step."""

    def __init__(self):
        self.decision_engine = 'rule-based demonstration'

    def execute_logic(self, processed_data):
        return {
            'orders': random.randint(50, 200),
            'inventory': random.uniform(0.6, 0.9),
            'shipments': random.randint(20, 80),
            'vehicles': random.randint(10, 30),
            'workload': random.uniform(0.5, 0.9),
            'available_workers': random.randint(20, 50),
            'demand_patterns': 'normal',
            'seasonal_factors': 1.0,
            'packing_tasks': self._generate_packing_tasks(),
        }

    def _generate_packing_tasks(self):
        tasks = []
        for i in range(random.randint(5, 15)):
            items = [
                (
                    random.randint(1, 6),
                    random.randint(1, 5),
                    random.randint(1, 4),
                )
                for _ in range(random.randint(3, 8))
            ]
            bin_dims = (
                random.randint(8, 15),
                random.randint(6, 12),
                random.randint(4, 8),
            )
            tasks.append(
                {
                    'id': f'task_{i}',
                    'items': items,
                    'bin_dims': bin_dims,
                    'priority': random.choice(['low', 'medium', 'high']),
                }
            )
        return tasks


class WarehouseManagementSystem:
    def __init__(self):
        self.inventory_levels = random.uniform(0.6, 0.9)

    def process_orders(self, orders, inventory):
        return {
            'processed_orders': orders,
            'inventory_update': inventory * 0.95,
        }

    def get_congestion_level(self):
        return random.uniform(0.3, 0.8)

    def get_inventory_levels(self):
        return self.inventory_levels


class TransportationManagementSystem:
    def __init__(self):
        self.vehicle_capacity = 1000

    def schedule_shipments(self, shipments, vehicles):
        return {'scheduled_shipments': min(shipments, vehicles * 10)}

    def get_available_capacity(self):
        return random.uniform(0.5, 0.9)


class LaborManagementSystem:
    def __init__(self):
        self.total_workers = 50

    def allocate_workers(self, workload, available_workers):
        return {
            'allocated_workers': min(available_workers, int(workload * 40))
        }

    def get_available_workers(self):
        return random.uniform(0.6, 1.0)


class DemandForecastingSystem:
    def __init__(self):
        self.demand_history = []

    def update_forecast(self, demand_patterns, seasonal_factors):
        return {'forecast_accuracy': random.uniform(0.85, 0.95)}

    def get_demand_pressure(self):
        return random.uniform(0.4, 0.8)

    def get_historical_demand(self):
        return [random.randint(100, 300) for _ in range(30)]


class IoTSensorNetwork:
    def __init__(self, config):
        self.sensor_count = config.get('sensors', 500)

    def get_sensor_readings(self):
        return {
            'temperature': 22.5,
            'humidity': 50.0,
            'motion_detected': True,
        }


class RFIDTrackingSystem:
    def __init__(self):
        self.readers = 50

    def track_items(self):
        return {'items_tracked': random.randint(1000, 5000)}


class AutomatedEquipmentSystem:
    def __init__(self):
        self.robots = 10
        self.conveyors = 20

    def get_status(self):
        return {'operational': 0.9, 'maintenance': 0.08, 'offline': 0.02}


class DemandPredictor:
    def __init__(self):
        self.model_accuracy = 0.92

    def update_model(self, data):
        self.model_accuracy = min(0.95, self.model_accuracy + 0.001)

    def predict_demand(self, horizon_hours=24):
        return [random.randint(80, 250) for _ in range(horizon_hours)]


class DisruptionPredictor:
    def __init__(self):
        self.alert_threshold = 0.7

    def update_model(self, data):
        return None

    def predict_disruptions(self, horizon_hours=24):
        return [
            {'time': i, 'probability': random.uniform(0.1, 0.3)}
            for i in range(horizon_hours)
        ]


class KPIMonitor:
    def __init__(self):
        self.kpi_history = []

    def calculate_kpis(self, timestamp):
        return {
            'orders_processed_per_hour': random.randint(80, 200),
            'space_utilization': random.uniform(0.75, 0.92),
            'labor_efficiency': random.uniform(0.80, 0.95),
            'cost_per_order': random.uniform(3.5, 6.5),
            'system_uptime': random.uniform(0.95, 0.99),
            'packing_quality': random.uniform(0.90, 0.98),
            'customer_satisfaction': random.uniform(0.85, 0.95),
        }


class ScenarioAnalyzer:
    """Summarize simulated KPI values without asserting causal effects."""

    def __init__(self):
        self.analysis_methods = ['descriptive summary']

    def analyze_results(self, results):
        if not results:
            return {'mean_kpis': {}, 'observation_count': 0}
        kpi_names = results[0]['kpis']
        mean_kpis = {
            name: float(np.mean([entry['kpis'][name] for entry in results]))
            for name in kpi_names
        }
        return {
            'mean_kpis': mean_kpis,
            'observation_count': len(results),
        }

In [ ]:
warehouse_config = {
    'name': 'E-Commerce Fulfillment Center',
    'area': 50000,
    'zones': 6,
    'sensors': 500,
    'conveyors': 20,
    'robots': 10,
    'daily_capacity': 50000,
}

print("=" * 60)
print("DIGITAL TWIN FOR 3D PALLET/CASE PACKING")
print("=" * 60)
digital_twin = DigitalTwin3DPacking(
    warehouse_config, simulation_hours=8
)

print("\nRunning the simulated operations...")
start_time = time.time()
performance_history = digital_twin.run_simulation()
elapsed_time = time.time() - start_time
print(f"\nExecution time: {elapsed_time:.2f} seconds")
print(f"KPI observations collected: {len(performance_history)}")

# The checks verify the recorded data structure and the geometric placement
# invariants used by the single-bin packing routine.
assert len(performance_history) == digital_twin.simulation_hours
for entry in performance_history:
    assert set(entry) == {'timestamp', 'kpis'}
    assert all(0 <= entry['kpis'][key] <= 1 for key in (
        'space_utilization', 'labor_efficiency', 'system_uptime',
        'packing_quality', 'customer_satisfaction'
    ))

sample_items = [(3, 2, 2), (4, 3, 3), (2, 4, 2)]
sample_bin_dims = (10, 8, 6)
sample_solution = digital_twin._greedy_packing(
    sample_items, sample_bin_dims
)
assert sample_solution['items_packed'] == len(
    sample_solution['positions']
)
assert len(sample_solution['positions']) == len(
    sample_solution['placed_item_indices']
)
for item_idx, position in zip(
    sample_solution['placed_item_indices'], sample_solution['positions']
):
    assert digital_twin._is_position_feasible(
        item_idx, position,
        {
            'positions': [
                p for p in sample_solution['positions']
                if p != position
            ],
            'placed_item_indices': [
                i for i in sample_solution['placed_item_indices']
                if i != item_idx
            ],
        },
        sample_items,
        sample_bin_dims,
    )

In [ ]:
# Plot the recorded time series and summarize the simulated KPI values.
digital_twin.visualize_performance()

In [ ]:
scenarios = {
    'High_Demand_Surge': {
        'daily_capacity': 75000,
        'robots': 15,
        'sensors': 750,
    },
    'Labor_Shortage': {
        'robots': 20,
        'conveyors': 30,
        'daily_capacity': 40000,
    },
    'Equipment_Failure': {
        'robots': 5,
        'conveyors': 10,
        'daily_capacity': 30000,
    },
}

print("\n" + "=" * 60)
print("RUNNING WHAT-IF SCENARIO SIMULATIONS")
print("=" * 60)
scenario_results = digital_twin.run_what_if_analysis(scenarios)

print("\n" + "=" * 60)
print("DESCRIPTIVE SCENARIO SUMMARIES")
print("=" * 60)
for scenario_name, result in scenario_results.items():
    print(f"\n{scenario_name}")
    print(f"  Configuration parameters: {result['parameters']}")
    print(
        "  KPI observations: "
        f"{result['analysis']['observation_count']}"
    )
    for metric, value in result['analysis']['mean_kpis'].items():
        if metric in {
            'space_utilization', 'labor_efficiency', 'system_uptime',
            'packing_quality', 'customer_satisfaction'
        }:
            display_value = f"{value:.1%}"
        elif metric == 'cost_per_order':
            display_value = f"${value:.2f} per order"
        else:
            display_value = f"{value:.1f} orders/hour"
        print(f"  Mean {metric.replace('_', ' ')}: {display_value}")

print(
    "\nThese are stochastic simulation summaries. The model does not "
    "estimate the causal effect of the configuration changes."
)

In [ ]:
sample_tasks = [
    {
        'id': 'urgent_order',
        'items': [(3, 2, 2), (4, 3, 3), (2, 4, 2)],
        'bin_dims': (10, 8, 6),
        'priority': 'high',
    },
    {
        'id': 'bulk_shipment',
        'items': [(5, 4, 3), (6, 3, 4), (4, 5, 3), (3, 4, 2)],
        'bin_dims': (12, 10, 8),
        'priority': 'medium',
    },
    {
        'id': 'standard_order',
        'items': [(2, 2, 2), (3, 3, 2), (4, 2, 3)],
        'bin_dims': (8, 6, 6),
        'priority': 'low',
    },
]

conditions_scenarios = [
    {
        'name': 'Normal Operations',
        'conditions': {
            'labor_availability': 0.8,
            'equipment_status': {'operational': 0.95},
            'congestion_level': 0.4,
            'demand_pressure': 0.5,
            'transport_capacity': 0.8,
            'inventory_levels': 0.7,
        },
    },
    {
        'name': 'Peak Demand',
        'conditions': {
            'labor_availability': 0.6,
            'equipment_status': {'operational': 0.85},
            'congestion_level': 0.8,
            'demand_pressure': 0.9,
            'transport_capacity': 0.5,
            'inventory_levels': 0.9,
        },
    },
    {
        'name': 'Labor Shortage',
        'conditions': {
            'labor_availability': 0.3,
            'equipment_status': {'operational': 0.9},
            'congestion_level': 0.6,
            'demand_pressure': 0.6,
            'transport_capacity': 0.7,
            'inventory_levels': 0.5,
        },
    },
]

print("\n" + "=" * 60)
print("DYNAMIC PACKING POLICY SELECTION")
print("=" * 60)
for scenario in conditions_scenarios:
    conditions = scenario['conditions']
    print(f"\n{scenario['name']}:")
    print(
        f"  Labor: {conditions['labor_availability']:.1%}, "
        f"congestion: {conditions['congestion_level']:.1%}, "
        f"demand pressure: {conditions['demand_pressure']:.1%}"
    )
    for task in sample_tasks:
        strategy = digital_twin._select_packing_strategy(task, conditions)
        result = digital_twin._pack_items(
            task['items'], task['bin_dims'], strategy
        )
        utilization = result.get('utilization')
        if utilization is None:
            bin_volume = np.prod(task['bin_dims'])
            packed_volume = sum(
                task['items'][i][0]
                * task['items'][i][1]
                * task['items'][i][2]
                for i in result['placed_item_indices']
            )
            utilization = packed_volume / bin_volume
        print(
            f"  Task {task['id']}: {strategy}; "
            f"{result['items_packed']}/{len(task['items'])} items packed; "
            f"volume utilization {utilization:.1%}"
        )

print(
    "\nThe policy names communicate intended priorities. The policies "
    "share the same underlying placement heuristic in this simulation."
)

In [ ]:
print("\n" + "=" * 60)
print("ILLUSTRATIVE SYSTEM-INTEGRATION METRIC COMPARISON")
print("=" * 60)

# These baseline values are explicit assumptions, not measurements from a
# separate control group. Only metrics generated by the simulation are
# compared; no unobserved error-rate improvement is assumed.
baseline_metrics = {
    'orders_per_hour': 120,
    'space_utilization': 0.75,
    'labor_efficiency': 0.82,
    'cost_per_order': 5.50,
    'customer_satisfaction': 0.85,
}
kpi_data = [entry['kpis'] for entry in digital_twin.performance_history]
integrated_metrics = {
    'orders_per_hour': float(np.mean([
        kpi['orders_processed_per_hour'] for kpi in kpi_data
    ])),
    'space_utilization': float(np.mean([
        kpi['space_utilization'] for kpi in kpi_data
    ])),
    'labor_efficiency': float(np.mean([
        kpi['labor_efficiency'] for kpi in kpi_data
    ])),
    'cost_per_order': float(np.mean([
        kpi['cost_per_order'] for kpi in kpi_data
    ])),
    'customer_satisfaction': float(np.mean([
        kpi['customer_satisfaction'] for kpi in kpi_data
    ])),
}

lower_is_better = {'cost_per_order'}
improvements = {}
for metric, baseline_value in baseline_metrics.items():
    integrated_value = integrated_metrics[metric]
    if metric in lower_is_better:
        improvement = (
            (baseline_value - integrated_value) / baseline_value * 100
        )
    else:
        improvement = (
            (integrated_value - baseline_value) / baseline_value * 100
        )
    improvements[metric] = improvement

print(
    f"{'Metric':<25} {'Assumed baseline':<19} "
    f"{'Simulation mean':<18} {'Relative change':<16}"
)
print("-" * 78)
for metric, baseline_value in baseline_metrics.items():
    integrated_value = integrated_metrics[metric]
    if metric in {
        'space_utilization', 'labor_efficiency', 'customer_satisfaction'
    }:
        baseline_text = f"{baseline_value:.1%}"
        integrated_text = f"{integrated_value:.1%}"
    elif metric == 'cost_per_order':
        baseline_text = f"${baseline_value:.2f}"
        integrated_text = f"${integrated_value:.2f}"
    else:
        baseline_text = f"{baseline_value:.1f} orders/hour"
        integrated_text = f"{integrated_value:.1f} orders/hour"
    print(
        f"{metric.replace('_', ' ').title():<25} "
        f"{baseline_text:<19} {integrated_text:<18} "
        f"{improvements[metric]:+.1f}%"
    )

# This extrapolation assumes the same 8-hour operating day on each of 365
# days. It is an arithmetic scenario, not a forecast of realized savings.
daily_orders = integrated_metrics['orders_per_hour'] * 8
daily_cost_difference = (
    baseline_metrics['cost_per_order']
    - integrated_metrics['cost_per_order']
) * daily_orders
annualized_cost_difference = daily_cost_difference * 365
print("\nIllustrative extrapolation under the stated operating assumption:")
print(f"  Orders per 8-hour day: {daily_orders:.0f}")
print(f"  Cost difference per day: ${daily_cost_difference:.2f}")
print(
    f"  Annualized cost difference: ${annualized_cost_difference:,.2f}"
)
print(
    "This comparison does not isolate system integration as a cause; "
    "baseline values are assumptions and simulation KPIs are stochastic."
)

## System-of-systems interpretation

The simulation organizes its components into four conceptual layers:

- **Physical layer:** modeled facility assets and generated sensor readings.
- **Connectivity layer:** a simplified representation of transmitting those readings.
- **Data-processing layer:** a pass-through simulation step with illustrative data-quality metadata.
- **Application layer:** generation of operational decisions and packing tasks.

The warehouse, transportation, labor, and demand classes provide separate interfaces for the modeled functions. Their presence demonstrates how system responsibilities can be separated in software; it does not imply that these components exchange production data or that the interfaces represent a complete enterprise integration.

### Policy selection and packing interpretation

The policy selector applies its conditions in order. For example, high demand pressure takes priority over labor availability, congestion, and transport capacity. The selected name determines which packing method is called, but the policies in this notebook share the same underlying greedy placement approach. The displayed volume utilization is the sum of packed item volumes divided by bin volume. It measures volume occupancy, not the quality of the arrangement, stability, load-bearing safety, or operational throughput.

### What-if analysis

Each configured scenario creates a new twin and runs a short simulation. The analyzer reports descriptive averages of the generated KPIs. Since the KPI generator does not use the scenario configuration to model performance effects, differences between scenario summaries are not evidence that the parameter changes caused those differences. A causal comparison would require an explicitly specified response model and a suitable experimental design.

### Limitations and natural extensions

This teaching model omits item rotation, multiple-bin allocation, support and stability constraints, weight limits, precedence constraints, labor and equipment capacity constraints, calibrated cost functions, and real sensor streams. It also does not estimate an optimal system-wide objective. These are modeling extensions that would require additional data and carefully specified constraints; they are not represented by the current simulation.